# Adult Income Classification

## Predicting whether annual income exceeds $50K

This project builds an end-to-end classification pipeline using the UCI Adult Income dataset. The goal is to predict whether an individual earns more than $50,000 per year while handling mixed data types, missing values, class imbalance, and hyperparameter optimization without data leakage.

### Project goals

- Explore the class distribution and prepare the data for modeling
- Engineer features that better represent education and capital income
- Build a leak-resistant preprocessing and modeling pipeline
- Compare a baseline model with randomized search and Optuna optimization
- Evaluate the final model using F1, precision, recall, and balanced accuracy

## 1. Setup

The analysis uses pandas and NumPy for data handling, scikit-learn for preprocessing and modeling, Matplotlib for visualization, and Optuna for Bayesian hyperparameter optimization.

Install Optuna separately when needed:

```bash
pip install optuna
```

In [20]:
from sklearn.base import clone

In [21]:
# General utilities
import os
import io
import time
import zipfile
import requests
from collections import Counter

# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display
 
# Data source
from sklearn.datasets import fetch_openml

 
# scikit-learn core tools 
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

 
# Import model 
from sklearn.ensemble import HistGradientBoostingClassifier
 
# Metrics
from sklearn.metrics import balanced_accuracy_score, classification_report
 
# Distributions for random search
from scipy.stats import loguniform, randint, uniform

# pandas dtypes helpers
from pandas.api.types import is_numeric_dtype, is_categorical_dtype
from pandas import CategoricalDtype

# Optuna Hyperparameter Search tool    (may need to be installed)
import optuna


# Misc

random_seed = 42

def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))




[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Load the dataset

The UCI Adult Income dataset contains demographic and employment-related attributes. The target identifies whether annual income is above or below $50K. Missing values represented by `?` are converted to `NaN` for consistent preprocessing.

In [22]:
# Load and clean
df = fetch_openml(name='adult', version=2, as_frame=True).frame

df.replace("?", np.nan, inplace=True)            # Some datasets use ? instead of Nan for missing data

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


## 3. Examine class balance

Because the higher-income class is less common, accuracy alone could be misleading. The project therefore uses stratified splits, balanced class weights, F1 score, and balanced accuracy.

In [23]:
print(df['class'].value_counts(normalize=True))

class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


The target is moderately imbalanced, so the modeling strategy explicitly compensates for the minority class during training and evaluation.

## 4. Feature engineering

The following transformations simplify redundant variables and make capital income easier for the model to learn:

- Remove `fnlwgt`, a survey-weight variable that is not a personal characteristic
- Keep `education-num` and remove the duplicate categorical `education` feature
- Combine capital gains and losses into `capital_net`
- Add a signed log transformation to reduce the extreme skew in capital values

In [24]:
# Drop the survey-weight column
df_eng = df.drop(columns=["fnlwgt"])

# Keep only the ordinal education feature
df_eng = df_eng.drop(columns=["education"])      # retain 'education-num'

# Combine capital gains and losses, add a log-scaled variant
df_eng["capital_net"]     = df_eng["capital-gain"] - df_eng["capital-loss"]
df_eng["capital_net_log"] = np.log1p(df_eng["capital_net"].clip(lower=0))
df_eng = df_eng.drop(columns=["capital-gain", "capital-loss"])

# check
df_eng.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   age              48842 non-null  int64   
 1   workclass        46043 non-null  category
 2   education-num    48842 non-null  int64   
 3   marital-status   48842 non-null  category
 4   occupation       46033 non-null  category
 5   relationship     48842 non-null  category
 6   race             48842 non-null  category
 7   sex              48842 non-null  category
 8   hours-per-week   48842 non-null  int64   
 9   native-country   47985 non-null  category
 10  class            48842 non-null  category
 11  capital_net      48842 non-null  int64   
 12  capital_net_log  48842 non-null  float64 
dtypes: category(8), float64(1), int64(4)
memory usage: 2.2 MB


## 5. Train-test split

The data is divided into an 80% training set and a 20% held-out test set. Stratification preserves the income-class proportions in both sets.

In [25]:

X = df_eng.drop(columns=["class"])
y = (df_eng["class"] == ">50K").astype(int)

# Split (with stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_seed,
    stratify=y                           # So same proportion of classes in train and test sets
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape,  y_test.shape)

Train: (39073, 12) (39073,)
Test : (9769, 12) (9769,)


## 6. Modeling pipeline

A `HistGradientBoostingClassifier` is paired with an `OrdinalEncoder` inside a scikit-learn `Pipeline`. Keeping preprocessing inside the pipeline prevents information from the validation folds from leaking into training during cross-validation.

In [26]:
# Define a baseline model 

HGBC_model = HistGradientBoostingClassifier(
    # tree structure and learning rate
    learning_rate=0.1,            # These 5 parameters are at defaults for our baseline training in Problem 1             
    max_leaf_nodes=31,            # but will be tuned by randomized search in Problem 2 and Optuna in Problem 3               
    max_depth=None,               
    min_samples_leaf=20,          
    l2_regularization=0.0,        

    # bins and iteration
    max_bins=255,                 # default
    max_iter=500,                 # high enough for early stopping
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.2,      # 20% monitored for early stopping
    tol=1e-7,                     # default tolerance for validation improvement

    # class imbalance
    class_weight="balanced",

    random_state=random_seed,
    verbose=0
)


In [27]:
enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",   # Allow unseen categories during transform
    unknown_value=-1,                     # Code for unseen categories
    encoded_missing_value=-2,             # Code for missing values (NaN)
    dtype=np.int64                        # Needed for HistGradientBoostingClassifier
)

# Categorical features
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

# Numeric features (everything that isn’t object / category)
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

preprocess = ColumnTransformer(
    [("cat", enc, cat_cols),
     ("num", "passthrough", num_cols)]
)

pipelined_model = Pipeline([
    ("prep", preprocess),
    ("gb",   HGBC_model)
])

## 7. Baseline cross-validation

The baseline pipeline is evaluated with five-fold stratified cross-validation using F1 score. F1 is appropriate here because it balances precision and recall for the less common `>50K` class.

In [28]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=random_seed)
# Estimate F1 across stratified folds
scores = cross_val_score(
    pipelined_model, X_train, y_train,
    cv=skf,
    scoring='f1'
)

# 3. Print mean and standard deviation, rounded to 4 decimal places
print(f"Mean F1 score: {round(scores.mean(), 4)}")
print(f"Standard deviation: {round(scores.std(), 4)}")

Mean F1 score: 0.7123
Standard deviation: 0.0035


In [ ]:
baseline_f1 = scores.mean()
print(f"Baseline mean F1: {baseline_f1:.4f}")


## 8. Randomized hyperparameter search

`RandomizedSearchCV` samples combinations of learning rate, tree size, leaf size, regularization, and iteration count. This provides a computationally efficient benchmark for tuning the gradient-boosting model.

In [62]:

# Define the parameter distributions
param_distributions = {
    "gb__learning_rate":      loguniform(1e-3, 0.3),   # log-uniform for learning rate
    "gb__max_leaf_nodes":     randint(16, 257),         # randint upper bound is exclusive, so 257 to include 256
    "gb__max_depth":          randint(2, 11),           # to include 10
    "gb__min_samples_leaf":   randint(10, 201),         # to include 200
    "gb__l2_regularization":  uniform(0.0, 2.0),        # uniform(loc, scale) -> range [0.0, 2.0]
}

# 2. Reuse the same StratifiedKFold setup from Problem 1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

# 3. Set up RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=pipelined_model,
    param_distributions=param_distributions,
    n_iter=20,                 # start small to prototype, bump to 50-100 later
    scoring="f1",
    cv=skf,
    random_state=random_seed,
    n_jobs=-1,                 # use all cores to speed things up
    verbose=1,
    refit=True
)

# Run the search
random_search.fit(X_train, y_train)

# 4. Show top 5 results in a neat table
results_df = pd.DataFrame(random_search.cv_results_)

top5 = (
    results_df
    .sort_values("rank_test_score")
    .head(5)
    .loc[:, ["mean_test_score", "std_test_score", "params"]]
    .reset_index(drop=True)
)
top5["mean_test_score"] = top5["mean_test_score"].round(4)
top5["std_test_score"] = top5["std_test_score"].round(4)

display(top5)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,mean_test_score,std_test_score,params
0,0.7120,0.0031,"{'gb__l2_regularization': 1.8299193510875615, ..."
1,0.7110,0.0028,"{'gb__l2_regularization': 1.6890676973563028, ..."
2,0.7106,0.0021,"{'gb__l2_regularization': 0.28573363584388156,..."
3,0.7102,0.0038,"{'gb__l2_regularization': 0.749080237694725, '..."
4,0.7101,0.0040,"{'gb__l2_regularization': 1.4580143360819746, ..."


In [ ]:
print(f"Randomized-search best F1: {random_search.best_score_:.4f}")
print("Best parameters:", random_search.best_params_)


## 9. Optuna optimization

Optuna performs guided hyperparameter search, using results from earlier trials to choose promising configurations. The same stratified cross-validation strategy and F1 objective are retained for a fair comparison.

In [34]:

# Reuse the same CV strategy as before
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

def objective(trial):
    # Same 5 hyperparameter ranges as Problem 2
    params = {
        "gb__learning_rate":     trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "gb__max_leaf_nodes":    trial.suggest_int("max_leaf_nodes", 16, 256),
        "gb__max_depth":         trial.suggest_int("max_depth", 2, 10),
        "gb__min_samples_leaf":  trial.suggest_int("min_samples_leaf", 10, 200),
        "gb__l2_regularization": trial.suggest_float("l2_regularization", 0.0, 2.0),
    }

    # Apply these params to a fresh clone of the pipeline
    model = clone(pipelined_model)
    model.set_params(**params)

    # Evaluate with the same CV setup as Problems 1 & 2
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="f1", n_jobs=-1)
    return scores.mean()

# 2. Set up and run the study
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_seed))
study.optimize(objective, n_trials=50, show_progress_bar=True)

# 3. Build a clean table of the top 5 trials
trials_df = study.trials_dataframe()
top5 = (
    trials_df
    .sort_values("value", ascending=False)
    .head(5)
    .loc[:, ["number", "value", "params_learning_rate", "params_max_leaf_nodes",
             "params_max_depth", "params_min_samples_leaf", "params_l2_regularization"]]
    .rename(columns={"value": "f1_score"})
    .reset_index(drop=True)
)
top5["f1_score"] = top5["f1_score"].round(4)

display(top5)

print("Best F1:", round(study.best_value, 4))
print("Best params:", study.best_params)

[I 2026-07-05 13:53:19,062] A new study created in memory with name: no-name-a6bbf5bf-7479-48d7-81b4-373222c5559a


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-05 13:53:24,887] Trial 0 finished with value: 0.6932940220816347 and parameters: {'learning_rate': 0.008468008575248327, 'max_leaf_nodes': 245, 'max_depth': 8, 'min_samples_leaf': 124, 'l2_regularization': 0.31203728088487304}. Best is trial 0 with value: 0.6932940220816347.
[I 2026-07-05 13:53:29,836] Trial 1 finished with value: 0.6757628077310386 and parameters: {'learning_rate': 0.0024345423962016913, 'max_leaf_nodes': 29, 'max_depth': 9, 'min_samples_leaf': 124, 'l2_regularization': 1.416145155592091}. Best is trial 0 with value: 0.6932940220816347.
[I 2026-07-05 13:53:38,432] Trial 2 finished with value: 0.6737028554087643 and parameters: {'learning_rate': 0.001124579825911934, 'max_leaf_nodes': 249, 'max_depth': 9, 'min_samples_leaf': 50, 'l2_regularization': 0.36364993441420124}. Best is trial 0 with value: 0.6932940220816347.
[I 2026-07-05 13:53:42,951] Trial 3 finished with value: 0.6702328589793822 and parameters: {'learning_rate': 0.002846526357761094, 'max_leaf_

,number,f1_score,params_learning_rate,params_max_leaf_nodes,params_max_depth,params_min_samples_leaf,params_l2_regularization
0,15,0.7122,0.158175,140,4,55,0.913087
1,41,0.7121,0.178322,64,5,27,0.668963
2,21,0.7121,0.170564,119,5,37,1.209517
3,28,0.7120,0.023486,163,7,10,1.083137
4,17,0.7119,0.126398,138,3,47,1.289851


Best F1: 0.7122
Best params: {'learning_rate': 0.15817490209956073, 'max_leaf_nodes': 140, 'max_depth': 4, 'min_samples_leaf': 55, 'l2_regularization': 0.9130867133019257}


In [ ]:
print(f"Optuna best F1: {study.best_value:.4f}")
print("Best parameters:", study.best_params)


## 10. Final test-set evaluation

The best Optuna configuration is fitted on the full training set and evaluated once on the untouched test set. Balanced accuracy measures performance across both classes, while the classification report shows class-specific precision, recall, and F1.

In [44]:

best_params = study.best_params
best_params_prefixed = {f"gb__{k}": v for k, v in best_params.items()}
final_model = clone(pipelined_model)
final_model.set_params(**best_params_prefixed)
final_model.fit(X_train, y_train)

# 3. Evaluate on the heldout test set
y_pred = final_model.predict(X_test)

# 4. Report metrics
print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))

bal_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced Accuracy: {round(bal_acc, 4)}")

              precision    recall  f1-score   support

       <=50K       0.95      0.82      0.88      7431
        >50K       0.60      0.87      0.71      2338

    accuracy                           0.83      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.83      0.84      9769

Balanced Accuracy: 0.8448


## 11. Results summary

In [ ]:
results_summary = pd.DataFrame({
    "Evaluation": [
        "Baseline cross-validation F1",
        "Randomized-search best F1",
        "Optuna best F1",
        "Test balanced accuracy",
    ],
    "Score": [
        scores.mean(),
        random_search.best_score_,
        study.best_value,
        bal_acc,
    ],
})

results_summary


## 12. Key takeaways

- The baseline gradient-boosting pipeline already performed strongly, and tuning produced only a small change in cross-validation F1.
- The final model achieved strong balanced accuracy and high recall for individuals earning more than $50K.
- Higher recall came with lower precision for the positive class, meaning the model captured most higher-income cases but also produced some false positives.
- The pipeline structure is a major strength of the project because it keeps encoding inside cross-validation and reduces leakage risk.
- A useful next step would be threshold tuning or calibration to explicitly manage the precision-recall tradeoff for a chosen business objective.